# Weak-lensing galaxy shape catalogue validation

## Global metacalibration

Contents.
- Metacalibration (global)
- Additive bias, ellipticity, magnitude distributions

> **_NOTE:_** Before running this notebook, set kernel to `main_set.ipynb'

In [ ]:
%matplotlib inline

import os
from uncertainties import ufloat

from lenspack.geometry.projections.gnom import radec2xy

In [ ]:
from sp_validation.survey import *
from sp_validation.basic import *
from sp_validation.util import *
from sp_validation.basic import *
from sp_validation.plots import *
from sp_validation.plot_style import *
from sp_validation.calibration import *

## metacalibration for galaxies

In [ ]:
gal_metacal = {}

for sh in shapes:
    print(f'{sh}:')
    gal_metacal[sh] = metacal(
        dd,
        m_gal[sh],
        prefix=sh.upper(),
        snr_min=gal_snr_min,
        snr_max=gal_snr_max,
        rel_size_min=gal_rel_size_min,
        verbose=verbose
    )

#### Extract quantities after calibration and cuts (in metacal)

In [ ]:
g_corr = {}
g_uncorr = {}
w = {}
mask = {}
ra = {}
dec = {}
ra_mean = {}
dec_mean = {}
mag = {}
snr = {}

for sh in shapes:
    g_corr[sh], g_uncorr[sh], w[sh], mask[sh] = (
        get_calibrated_quantities(gal_metacal[sh])
    )

    # coordinates
    ra[sh] = dd['XWIN_WORLD'][m_gal[sh]][mask[sh]]
    dec[sh] = dd['YWIN_WORLD'][m_gal[sh]][mask[sh]]
    ra_mean[sh] = np.mean(ra[sh])
    dec_mean[sh] = np.mean(dec[sh])

    # magnitude, from SExtractor
    mag[sh] = dd['MAG_AUTO'][m_gal[sh]][mask[sh]]

if 'ngmix' in shapes:
    # signal-to-noise ratio, only for ngmix, from fitted flux and error
    snr['ngmix'] = (
        dd['NGMIX_FLUX_NOSHEAR'][m_gal['ngmix']][mask['ngmix']] 
        / dd['NGMIX_FLUX_ERR_NOSHEAR'][m_gal['ngmix']][mask['ngmix']]
    )

#### Compute coordinates for projection and spatial binning (needed later)

In [ ]:
x = {}
y = {}

for sh in shapes:
    # Projection all objects from spherical to Cartesian coordinates
    x[sh], y[sh] =  radec2xy(ra_mean[sh], dec_mean[sh], ra[sh], dec[sh])

#### Compute field size

In [ ]:
min_x = {}
max_x = {}
min_y = {}
max_y = {}
size_x_deg = {}
size_y_deg = {}

for sh in shapes:
    # Define mix, max and size
    min_x[sh] = np.min(x[sh])
    max_x[sh] = np.max(x[sh])
    min_y[sh] = np.min(y[sh])
    max_y[sh] = np.max(y[sh])

    size_x_deg[sh] = np.rad2deg(max_x[sh] - min_x[sh])
    size_y_deg[sh] = np.rad2deg(max_y[sh] - min_y[sh])

    print(
        f'{sh}: Field size in projected coordinates is (x, y) '
        + f'= ({size_x_deg[sh]:.2f}, {size_y_deg[sh]:.2f}) deg'
    )

In [ ]:
# Get coordinates for all objects
ra['all'] = dd['XWIN_WORLD']
dec['all'] = dd['YWIN_WORLD']

shapes_all = []
for sh in shapes:
    shapes_all.append(sh)
shapes_all.append('all')

In [ ]:
# Number density
n_gal = {}
n_gal_sm = {}

for sh in shapes:
    n_gal[sh] = len(w[sh])
    n_gal_sm[sh] = len(np.where(m_gal[sh])[0])

    print_stats(sh, stats_file, verbose=verbose)
    print_stats(
        f'Number of galaxies after metacal = {n_gal[sh]}/{n_gal_sm[sh]} '
        + f'= {n_gal[sh] / n_gal_sm[sh] * 100:.1f}%',
        stats_file,
        verbose=verbose
    )
    print_stats(
        f'Galaxy density = {n_gal[sh] / area_amin2:.2f} gal/arcmin2',
        stats_file,
        verbose=verbose
    )

#### Plot spatial distribution of objects

In [ ]:
x_label = 'R.A. [deg]'
y_label = 'DEC [deg]'
cbar_label_base = 'Density [$A_{\\rm pix}^{-1}$]'

In [ ]:
# Galaxies
Apix = 1 # [arcmin^2]
cbar_label = '{}, $A_{{\\rm pix}} \\approx {:.1g}$ arcmin$^2$'.format(cbar_label_base, Apix)
n_grid = int(np.sqrt(area_amin2) / Apix)
if verbose:
    print('Number of pixels = {}^2'.format(n_grid))
    
for sh in shapes_all:
    title = f'Galaxies ({sh})'
    out_path = f'{plot_dir}/galaxy_number_count_{sh}'
    plot_spatial_density(
        ra[sh],
        dec[sh],
        title,
        x_label,
        y_label,
        cbar_label,
        out_path,
        n_grid=n_grid,
        verbose=verbose
    )

#### Plot galaxy signal-to-noise distribution

In [ ]:
x_label = 'SNR'
y_label = 'Frequency'
density = True
x_range = (0, 200)
n_bin = 500
x_cut = gal_snr_min

for sh in shapes:
    print_stats(sh, stats_file, verbose=verbose)

    labels = []
    if sh == 'ngmix':
    # Do not apply `mask_ns`, so use all galaxies
        xs = [
            dd['NGMIX_FLUX_NOSHEAR'][m_gal[sh]] / dd['NGMIX_FLUX_ERR_NOSHEAR'][m_gal[sh]],
            dd['SNR_WIN'][m_gal[sh]]
        ]
        labels.append([f'{sh} $F/\\sigma(F)$'])

    elif sh == 'galsim':
        xs = [
            dd['SNR_WIN'][m_gal[sh]]
        ]
  
    labels.append(f'{sh} SExtractor SNR')

    title = f'Galaxies ({sh})'

    out_name = f'hist_SNR_{sh}.pdf'
    out_path = os.path.join(plot_dir, out_name)

    plot_histograms(
        xs,
        labels,
        title,
        x_label,
        y_label,
        x_range,
        n_bin,
        out_path,
        vline_x=[x_cut],
        vline_lab=[f'SNR = {x_cut}']
    )

## Metacalibration for stars

In [ ]:
star_metacal = {}
for sh in shapes:
    star_metacal[sh] = metacal(dd[ind_star], m_star[sh], masking_type='star')

#### Number density

In [ ]:
# mask for 'no shear' images
mask_ns_stars = {}
n_star = {}

for sh in shapes:
    mask_ns_stars[sh] = star_metacal[sh].mask_dict['ns']
    n_star[sh] = len(star_metacal[sh].ns['g1'][mask_ns_stars[sh]])

    print_stats(f'{sh}:', stats_file, verbose=verbose)
    print_stats(f'Number of stars = {n_star[sh]}', stats_file, verbose=verbose)
    print_stats('Star density = {:.2f} stars/deg2'.format(n_star[sh] / area_deg2), stats_file, verbose=verbose)

## Additive bias

In [ ]:
n_jack = 500

print_stats('additive bias', stats_file, verbose=verbose)

In [ ]:
c = {}
c_err = {}

for sh in shapes:
    print_stats(f'{sh}:', stats_file, verbose=verbose)
    
    c[sh] = np.zeros(2)
    c_err[sh] = np.zeros(2)

    for comp in (0, 1):
        c[sh][comp], c_err[sh][comp] = jackknif_weighted_average(
            g_corr[sh][comp],
            w[sh],
            remove_size=0.05,
            n_realization=n_jack
        )
        c_dc = ufloat(c[sh][comp],c_err[sh][comp])
        print_stats(f'c_{comp+1} = {c_dc:.2eP}', stats_file, verbose=verbose)

## Get quantities calibrated for both multiplicative and additive biase

In [ ]:
g_corr_mc = {}

for sh in shapes:
    g_corr_mc[sh] = np.zeros_like(g_corr[sh])
    for comp in (0, 1):
        g_corr_mc[sh][comp] = g_corr[sh][comp] - c[sh][comp]

## Response matrix

### Mean

In [ ]:
R_shear = {}

for sh in shapes:
    print_stats(f'{sh} galaxies:', stats_file, verbose=verbose)

    print_stats('total response matrix:', stats_file, verbose=verbose)
    rs = np.array2string(gal_metacal[sh].R)
    print_stats(rs, stats_file, verbose=verbose)

    print_stats('shear response matrix:', stats_file, verbose=verbose)
    R_shear[sh] = np.mean(gal_metacal[sh].R_shear, 2)
    rs = np.array2string(R_shear[sh])
    print_stats(rs, stats_file, verbose=verbose)

    print_stats('selection response matrix:', stats_file, verbose=verbose)
    rs = np.array2string(gal_metacal[sh].R_selection)
    print_stats(rs, stats_file, verbose=verbose)

In [ ]:
R_shear_stars = {}

for sh in shapes:
    print_stats(f'{sh} stars:', stats_file, verbose=verbose)

    print_stats('total response matrix:', stats_file, verbose=verbose)
    rs = np.array2string(star_metacal[sh].R)
    print_stats(rs, stats_file, verbose=verbose)

    print_stats('shear response matrix:', stats_file, verbose=verbose)
    R_shear_stars[sh] = np.mean(star_metacal[sh].R_shear, 2)
    rs = np.array2string(R_shear_stars[sh])
    print_stats(rs, stats_file, verbose=verbose)

    print_stats('selection response matrix:', stats_file, verbose=verbose)
    rs = np.array2string(star_metacal[sh].R_selection)
    print_stats(rs, stats_file, verbose=verbose)

### Plot distribution of response matrix elements

In [ ]:
x_label = 'response matrix element'
y_label = 'Frequency'
x_range = (-3, 3)
n_bin = 500

In [ ]:
colors = ['blue', 'red','blue', 'red']
linestyles = ['-', '-', ':', ':']

In [ ]:
labels = [
    '$R_{11}$ galaxies',
    '$R_{22}$ galaixes',
    '$R_{11}$ stars',
    '$R_{22}$ stars'
]

for sh in shapes:

    xs = [
        gal_metacal[sh].R_shear[0,0],
        gal_metacal[sh].R_shear[1,1],
        star_metacal[sh].R_shear[0,0],
        star_metacal[sh].R_shear[1,1]
    ]
    title = sh
    
    out_name = f'R_{sh}_diag.pdf'
    out_path = os.path.join(plot_dir, out_name)
    
    plot_histograms(
        xs,
        labels,
        title,
        x_label,
        y_label,
        x_range,
        n_bin,
        out_path,
        colors=colors,
        linestyles=linestyles
    )

In [ ]:
labels = [
    '$R_{12}$ galaxies',
    '$R_{21}$ galaixes',
    '$R_{12}$ stars',
    '$R_{21}$ stars'
]

for sh in shapes:

    xs = [gal_metacal[sh].R_shear[0,1],
          gal_metacal[sh].R_shear[1,0],
          star_metacal[sh].R_shear[0,1],
          star_metacal[sh].R_shear[1,0]
         ]
    title = sh
    out_name = f'R_{sh}_offdiag.pdf'
    out_path = os.path.join(plot_dir, out_name)

    plot_histograms(
        xs,
        labels,
        title,
        x_label,
        y_label,
        x_range, 
        n_bin,
        out_path,
        colors=colors,
        linestyles=linestyles
    )

## Ellipticities

In [ ]:
x_label = 'ellipticity'
y_label = 'Frequency'
x_range = (-1, 1)
n_bin = 500

labels = ['$e_1$', '$e_2$']
colors = ['blue', 'red']
linestyles = ['-', '-'] 

In [ ]:
for sh in shapes:

    xs = [g_corr[sh][0], g_corr[sh][1]]
    weights = [w[sh]] * 2

    title = f'{sh} galaxies'
    out_name = f'ell_gal_{sh}.pdf'
    out_path = os.path.join(plot_dir, out_name)

    plot_histograms(
        xs, 
        labels, 
        title, 
        x_label, 
        y_label, 
        x_range, 
        n_bin,
        out_path,
        weights=weights, 
        colors=colors, 
        linestyles=linestyles
    )

In [ ]:
for sh in shapes:

    xs = [star_metacal[sh].ns['g1'][mask_ns_stars[sh]], star_metacal[sh].ns['g2'][mask_ns_stars[sh]]]
    weights = [star_metacal[sh].ns['w'][mask_ns_stars[sh]]] * 2

    title = f'{sh} stars'
    out_name = f'ell_stars_{sh}.pdf'
    out_path = os.path.join(plot_dir, out_name)

    plot_histograms(
        xs, 
        labels, 
        title, 
        x_label, 
        y_label, 
        x_range, 
        n_bin, 
        out_path,
        weights=weights, 
        colors=colors, 
        linestyles=linestyles
    )

In [ ]:
x_range = (-0.15, 0.15)
n_bin = 250

In [ ]:
for sh in shapes:

    key = key_PSF_ell[sh]
    xs = [
        dd[key][:,0][mask_ns_stars[sh]],
        dd[key][:,1][mask_ns_stars[sh]]
    ]
    title = f'{sh} PSF'
    out_name = f'ell_PSF_{sh}.pdf'
    out_path = os.path.join(plot_dir, out_name)

    plot_histograms(
        xs, 
        labels, 
        title, 
        x_label, 
        y_label, 
        x_range, 
        n_bin, 
        out_path,      
        colors=colors, 
        linestyles=linestyles
    )

## Magnitudes

In [ ]:
x_label = '$r$-band magnitude'
y_label = 'Frequency'
x_range = (19.8, 25.5)
n_bin = 500

colors = ['blue', 'red']
linestyles = ['-', '-']

title = 'galaxies'
out_name = 'mag_gal.pdf'
out_path = os.path.join(plot_dir, out_name)

In [ ]:
labels = []
xs = []

for sh in shapes:
    labels.append(sh)
    xs.append(dd['MAG_AUTO'][m_gal[sh]][mask[sh]])

plot_histograms(
    xs, 
    labels, 
    title, 
    x_label, 
    y_label, 
    x_range, 
    n_bin, 
    out_path,             
    colors=colors, 
    linestyles=linestyles
)

## Ellipticity dispersion

In [ ]:
for sh in shapes:
    print_stats(f'{sh}', stats_file, verbose=verbose)

    sig_eps = np.sqrt(np.var(g_corr[sh][0]) + np.var(g_corr[sh][1]))
    print_stats('Dispersion of complex ellipticity = {:.3f}' \
                ''.format(sig_eps), stats_file, verbose=verbose)
    print_stats('Dispersion of (average) single-component ellipticity = {:.3f} = {:.3f} / sqrt(2)' \
                ''.format(sig_eps /  np.sqrt(2), sig_eps), stats_file, verbose=verbose)